In [2]:
import pandas as pd
import os
import zipfile
import numpy as np
import torch
from torch.utils.data import Dataset
import timm
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [3]:
batch_size = 8

In [4]:
DATASET_PATH = "jaguar-re-id.zip"
EXTRACT_PATH = "data"

if not os.path.exists(EXTRACT_PATH):
    print("Разархивирование датасета...")
    with zipfile.ZipFile(DATASET_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print(f"Датасет разархивирован в {EXTRACT_PATH}")
else:
    print(f"Датасет уже находится в {EXTRACT_PATH}")

Датасет уже находится в data


In [5]:
path_train = 'data/train.csv'
path_test = 'data/test.csv'
train = pd.read_csv(path_train)
test = pd.read_csv(path_test)

In [6]:
encoder = LabelEncoder()

In [7]:
train['ground_truth'] = encoder.fit_transform(train['ground_truth'])

In [8]:
train.head()

,filename,ground_truth
0,train_0001.png,0
1,train_0002.png,0
2,train_0003.png,0
3,train_0004.png,1
4,train_0005.png,1


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    train['filename'], train['ground_truth'], test_size=0.15, random_state=12)

In [10]:
transform = T.Compose([T.Resize(size=(384, 384)),
                              T.ToTensor(), 
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

In [11]:
class CustomDataset(Dataset):
    def __init__(self, X=X_train, y=y_train, img_dir='data/train/train'):
        self.X = X
        self.y = y
        self.img = img_dir
        self.transform = T.Compose([T.Resize(size=(384, 384)),
                              T.ToTensor(), 
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])]) 
         
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        img_path = self.X.iloc[idx]
        img = Image.open(self.img + '/' + img_path).convert('RGB')
        return {'x': self.transform(img), 'y': self.y.iloc[idx]}

In [12]:
dataset_1 = CustomDataset()

In [ ]:
dataset_test = CustomDataset(X_test, y_test, 'data/test/test')

In [13]:
dataset_1.__len__()

1610

In [14]:
dataset_1.__getitem__(idx=1123)

{'x': tensor([[[ 0.2863,  0.3176,  0.3725,  ..., -0.1294, -0.1059, -0.0824],
          [ 0.0353,  0.1216,  0.2235,  ..., -0.2078, -0.1765, -0.0431],
          [-0.0980, -0.0824, -0.0353,  ..., -0.3255, -0.1765,  0.0039],
          ...,
          [ 0.1686,  0.0902,  0.0667,  ..., -0.2549, -0.2078, -0.2078],
          [ 0.1529,  0.1294,  0.1373,  ..., -0.2863, -0.1843, -0.1529],
          [ 0.1373,  0.1216,  0.1294,  ..., -0.2392, -0.1922, -0.1843]],
 
         [[ 0.2078,  0.2392,  0.2941,  ..., -0.2078, -0.1843, -0.1608],
          [-0.0667,  0.0118,  0.1137,  ..., -0.2863, -0.2549, -0.1216],
          [-0.2314, -0.2078, -0.1608,  ..., -0.4039, -0.2549, -0.0824],
          ...,
          [ 0.1373,  0.0588,  0.0353,  ..., -0.3412, -0.2941, -0.2941],
          [ 0.1137,  0.0902,  0.0980,  ..., -0.3725, -0.2706, -0.2392],
          [ 0.1059,  0.0824,  0.0902,  ..., -0.3255, -0.2784, -0.2706]],
 
         [[ 0.1373,  0.1843,  0.2627,  ..., -0.2235, -0.2000, -0.1765],
          [-0.1529, -0.

In [15]:
from transformers import DefaultDataCollator

In [16]:
data_collator = DefaultDataCollator()

In [17]:
train_dataloader = DataLoader(dataset_1, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

In [ ]:
test_dataloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

In [18]:
for batch in train_dataloader:
    break
print(batch['x'], batch['y'])


tensor([[[[-0.5137, -0.6706, -0.7490,  ..., -0.6549, -0.6314, -0.6235],
          [-0.5294, -0.6471, -0.7490,  ..., -0.6549, -0.6314, -0.6078],
          [-0.5686, -0.6235, -0.7490,  ..., -0.6549, -0.6314, -0.6157],
          ...,
          [-0.4353, -0.4275, -0.4118,  ..., -0.0510, -0.2000, -0.3490],
          [-0.4588, -0.4510, -0.4353,  ...,  0.0745,  0.0275, -0.1373],
          [-0.4588, -0.4588, -0.4431,  ...,  0.1686,  0.1294, -0.0039]],

         [[-0.4431, -0.6078, -0.6941,  ..., -0.4902, -0.4745, -0.4745],
          [-0.4667, -0.5922, -0.6941,  ..., -0.4902, -0.4745, -0.4510],
          [-0.5294, -0.5686, -0.6941,  ..., -0.4824, -0.4667, -0.4431],
          ...,
          [-0.4588, -0.4510, -0.4431,  ..., -0.0824, -0.2314, -0.3882],
          [-0.4745, -0.4667, -0.4510,  ...,  0.0431, -0.0039, -0.1686],
          [-0.4745, -0.4745, -0.4588,  ...,  0.1373,  0.0980, -0.0353]],

         [[-0.3569, -0.5294, -0.6392,  ..., -0.5059, -0.4824, -0.4510],
          [-0.3961, -0.5216, -

In [19]:
batch.items()

dict_items([('x', tensor([[[[-0.5137, -0.6706, -0.7490,  ..., -0.6549, -0.6314, -0.6235],
          [-0.5294, -0.6471, -0.7490,  ..., -0.6549, -0.6314, -0.6078],
          [-0.5686, -0.6235, -0.7490,  ..., -0.6549, -0.6314, -0.6157],
          ...,
          [-0.4353, -0.4275, -0.4118,  ..., -0.0510, -0.2000, -0.3490],
          [-0.4588, -0.4510, -0.4353,  ...,  0.0745,  0.0275, -0.1373],
          [-0.4588, -0.4588, -0.4431,  ...,  0.1686,  0.1294, -0.0039]],

         [[-0.4431, -0.6078, -0.6941,  ..., -0.4902, -0.4745, -0.4745],
          [-0.4667, -0.5922, -0.6941,  ..., -0.4902, -0.4745, -0.4510],
          [-0.5294, -0.5686, -0.6941,  ..., -0.4824, -0.4667, -0.4431],
          ...,
          [-0.4588, -0.4510, -0.4431,  ..., -0.0824, -0.2314, -0.3882],
          [-0.4745, -0.4667, -0.4510,  ...,  0.0431, -0.0039, -0.1686],
          [-0.4745, -0.4745, -0.4588,  ...,  0.1373,  0.0980, -0.0353]],

         [[-0.3569, -0.5294, -0.6392,  ..., -0.5059, -0.4824, -0.4510],
          [-

In [20]:
model = timm.create_model("hf-hub:BVRA/MegaDescriptor-L-384", pretrained=True, num_classes=31)

In [21]:
from transformers import TrainingArguments

# Показываем потери при обучении для каждой эпохи
logging_steps = train.shape[0] // batch_size
model_name = 're_identification_v1'

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    fp16=True,
    logging_steps=logging_steps,
)

In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_1
)

In [23]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [24]:
from transformers import get_scheduler

In [25]:
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1010


In [26]:
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

In [27]:
torch.cuda.empty_cache()

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.to('cuda')
model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        data = batch
        X = data['x'].to('cuda')
        y = data['y'].to('cuda')
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update()
        print(loss.item())
    
    torch.cuda.empty_cache()

In [ ]:
criterion(outputs, batch['y'])

tensor(3.7480, grad_fn=<NllLossBackward0>)